# 13b. Serving LLMs at Scale with vLLM

**Tier:** Building with LLMs
**Estimated time:** 55 minutes
**Prerequisites:** 04 (attention), 11 (speculative decoding), 13 (context windows & KV cache)
**Source material:** Kwon et al., *"Efficient Memory Management for Large Language Model Serving with PagedAttention"* (SOSP 2023) — the vLLM paper; vLLM docs (https://docs.vllm.ai); Stanford LLM curriculum, Lecture 3 (PagedAttention, continuous batching) — https://x.com/ajitcodes/status/2057043965317165490

## What You'll Learn
- Why the **KV cache** (notebook 13) — not the model weights — is the real bottleneck when you serve an LLM to many users at once
- How **PagedAttention** borrows operating-system virtual-memory paging to push GPU memory utilization from ~20–40% up to ~96%
- How **continuous batching** keeps the GPU full instead of letting short requests wait on long ones, for up to an order-of-magnitude more throughput

## Why This Matters
Everything before this notebook ran *one* prompt at a time on your laptop. Production is the opposite: thousands of concurrent users sharing one expensive GPU. vLLM is the most widely deployed open-source inference engine precisely because it solved the memory-fragmentation problem that made naive serving waste 60–80% of a GPU. Understanding *why* it's fast is the difference between renting four GPUs and renting one.


## The serving problem: the KV cache is the bottleneck, not the weights

In notebook 13 you saw that the **KV cache** stores the Key/Value vectors for every token so the model never recomputes them. On your laptop, serving one request, that cache is small and you never think about it. In production it dominates everything.

Here's the analogy. Think of GPU memory as a **hotel**. The model weights are the lobby and staff — a fixed cost you pay once, shared by everyone. Each *active request* is a **guest** who needs rooms (KV-cache memory) that grow as their conversation gets longer. The naive way to serve requests is the way a bad hotel books rooms: when a guest checks in, you reserve them the *entire top floor* up front in case their stay runs long (i.e. reserve `max_sequence_length` of contiguous memory per request). Most guests use three rooms and leave; the rest of the floor sits empty but unbookable. The hotel "fills up" at 30% real occupancy.

That's exactly what happened before vLLM. Two kinds of waste:
- **Internal fragmentation:** you reserved 2048 tokens of cache but the request only used 200 — the other 1848 slots are reserved-but-empty.
- **External fragmentation:** the leftover gaps between contiguous reservations are too small to fit a new request, even though the total free memory would be plenty.

Measured utilization in pre-vLLM systems was **20–40%**. PagedAttention fixes this the same way an OS fixed it for RAM in the 1970s: stop demanding one big contiguous block, hand out small fixed-size **pages** on demand instead. We'll build up to that — first, let's see the bottleneck with real numbers.


In [ ]:
import os, math
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

import numpy as np

# This notebook is fully offline-safe: every simulation below runs on CPU with no
# GPU and no model download. The only optional live piece is the very last section,
# which talks to a real vLLM server IF you happen to have one running locally.
def vllm_server_up(base_url="http://localhost:8000/v1", timeout=0.5):
    """Return True only if an OpenAI-compatible vLLM server answers locally."""
    import urllib.request, urllib.error
    try:
        urllib.request.urlopen(base_url.replace("/v1", "/health"), timeout=timeout)
        return True
    except Exception:
        return False

HAS_VLLM_SERVER = vllm_server_up()
print(f"Local vLLM server detected: {HAS_VLLM_SERVER}")
print("(That's expected to be False here — the serving section will explain instead of call.)")


## Real numbers: how much KV cache does one token actually cost?

The KV cache size per token is fixed by the model's shape. For a standard multi-head attention model:

$$\text{bytes per token} = 2 \times n_{\text{layers}} \times d_{\text{model}} \times \text{bytes per value}$$

The leading **2** is for storing both K *and* V. We multiply by the number of layers (every layer keeps its own cache) and the hidden size $d_{\text{model}}$ (= heads × head-dim), in your weight precision (fp16 = 2 bytes).

Let's compute it for a few real models and see how many requests fit on one GPU.


In [ ]:
DTYPE_BYTES = 2  # fp16 / bf16

# (name, num_layers, d_model, params_billion)
MODELS = [
    ("Llama-2-7B",   32, 4096,  7),
    ("OPT-13B",      40, 5120, 13),
    ("Llama-2-70B",  80, 8192, 70),  # note: real 70B uses GQA, which shrinks this further
]

def kv_bytes_per_token(num_layers, d_model, dtype_bytes=DTYPE_BYTES):
    return 2 * num_layers * d_model * dtype_bytes

print(f"{'model':14s} {'KV/token':>10s} {'weights':>9s} {'A100-40GB free for KV':>22s} {'max tokens cached':>18s}")
A100_GB = 40
for name, L, d, params_b in MODELS:
    per_tok = kv_bytes_per_token(L, d)
    weights_gb = params_b * DTYPE_BYTES            # ~2 bytes/param in fp16
    free_gb = A100_GB - weights_gb - 2             # minus weights, minus ~2GB overhead
    max_tokens = max(free_gb, 0) * 1e9 / per_tok
    print(f"{name:14s} {per_tok/1024:8.0f}KB {weights_gb:7.0f}GB {free_gb:20.0f}GB {max_tokens:18,.0f}")


Notice OPT-13B costs **~800 KB per token** — exactly the figure from the vLLM paper. With weights eating most of a 40 GB A100, you're left with room for only ~15,000 *total* cached tokens across all users. If you reserve `max_sequence_length = 2048` per request the naive way, that's room for just **~7 concurrent requests** — on a GPU that costs thousands of dollars a month. That tiny number is the whole motivation for what follows.

## Fragmentation: where the memory actually goes

Now let's *measure* the waste. We simulate a realistic batch of requests whose actual lengths vary (most short, a few long), and compare two allocators:

- **Naive contiguous:** reserve `max_len` for every request, regardless of how much it uses.
- **Paged (vLLM-style):** hand out fixed-size **blocks** (vLLM's default is 16 tokens) on demand, rounding each request up to the nearest block.


In [ ]:
rng = np.random.default_rng(7)
N_REQUESTS = 256
MAX_LEN = 2048        # the context length we'd have to reserve up front
BLOCK_SIZE = 16       # vLLM's default KV-cache block (page) size, in tokens

# Realistic skew: most chat requests are short, a long tail runs long.
# A lognormal clipped to [16, MAX_LEN] captures that shape well.
actual_lengths = np.clip(rng.lognormal(mean=5.9, sigma=0.85, size=N_REQUESTS),
                         16, MAX_LEN).astype(int)

used_tokens = actual_lengths.sum()

# Naive: every request reserves the full MAX_LEN of contiguous cache.
naive_reserved = N_REQUESTS * MAX_LEN

# Paged: each request rounds up only to the next whole block.
blocks_per_request = np.ceil(actual_lengths / BLOCK_SIZE).astype(int)
paged_reserved = (blocks_per_request * BLOCK_SIZE).sum()

naive_util = used_tokens / naive_reserved
paged_util = used_tokens / paged_reserved

print(f"Requests: {N_REQUESTS} | mean length {actual_lengths.mean():.0f} tokens "
      f"(min {actual_lengths.min()}, max {actual_lengths.max()})")
print(f"Tokens actually used        : {used_tokens:,}")
print(f"Naive contiguous reserved   : {naive_reserved:,}  -> utilization {naive_util:6.1%}")
print(f"Paged (block={BLOCK_SIZE}) reserved : {paged_reserved:,}  -> utilization {paged_util:6.1%}")
print(f"\nSame GPU memory fits ~{paged_reserved and naive_reserved/paged_reserved:.1f}x more requests with paging.")


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Left: memory utilization, naive vs paged
labels = ["Naive\n(reserve max_len)", "PagedAttention\n(16-token blocks)"]
utils = [naive_util * 100, paged_util * 100]
bars = ax1.bar(labels, utils, color=["#C44E52", "#55A868"])
ax1.set_ylabel("KV-cache memory utilization (%)")
ax1.set_title("Where the GPU memory goes")
ax1.set_ylim(0, 100)
for b, u in zip(bars, utils):
    ax1.text(b.get_x() + b.get_width()/2, u + 2, f"{u:.0f}%", ha="center", fontweight="bold")

# Right: distribution of request lengths vs the reserved ceiling
ax2.hist(actual_lengths, bins=40, color="#4C72B0", alpha=0.85)
ax2.axvline(MAX_LEN, color="#C44E52", linestyle="--", label=f"reserved ceiling = {MAX_LEN}")
ax2.set_xlabel("Actual request length (tokens)")
ax2.set_ylabel("# requests")
ax2.set_title("Most requests are short — the ceiling is mostly air")
ax2.legend()
plt.tight_layout(); plt.show()


*Left: reserving `max_len` per request strands most of the GPU as reserved-but-empty cache; paging gives it nearly all back. Right: the reserved ceiling (red) towers over the actual length of almost every request — that gap is the internal fragmentation paging eliminates.*

## PagedAttention: a block table, exactly like OS virtual memory

The fix is the same abstraction your operating system uses for RAM. A process *thinks* it has one contiguous address space (logical pages), but the OS maps those pages to scattered physical frames through a **page table**. vLLM does this for the KV cache:

- The KV cache is cut into fixed-size **blocks** (pages) of `BLOCK_SIZE` tokens.
- Each request keeps a **block table**: logical block 0, 1, 2… → physical block numbers, which can live *anywhere* in GPU memory, non-contiguously.
- A new block is allocated only when the current one fills up — so a request never reserves more than one block of slack.

This non-contiguous mapping unlocks a bonus the hotel analogy can't: **sharing**. If two requests share a prefix (the same long system prompt — notebook 13's prompt caching, notebook 16's "cold" layer), their block tables can point at the *same physical blocks*, copying-on-write only when their content diverges. Let's build a miniature allocator to make the mapping concrete.


In [ ]:
class PagedKVCache:
    """A toy PagedAttention allocator: fixed-size physical blocks + per-request block tables."""
    def __init__(self, num_physical_blocks, block_size=16):
        self.block_size = block_size
        self.free_blocks = list(range(num_physical_blocks))   # the physical "frames"
        self.block_tables = {}                                # request_id -> [physical block ids]
        self.ref_count = {}                                   # physical block -> # requests sharing it

    def _alloc_block(self):
        if not self.free_blocks:
            raise MemoryError("out of KV blocks — vLLM would preempt/evict a request here")
        blk = self.free_blocks.pop(0)
        self.ref_count[blk] = 1
        return blk

    def append_tokens(self, request_id, num_new_tokens):
        """Grow a request's cache by num_new_tokens, allocating blocks only as needed."""
        table = self.block_tables.setdefault(request_id, [])
        used_in_last = (self._logical_len.get(request_id, 0)) % self.block_size if hasattr(self, "_logical_len") else 0
        if not hasattr(self, "_logical_len"):
            self._logical_len = {}
        current = self._logical_len.get(request_id, 0)
        target = current + num_new_tokens
        needed_blocks = math.ceil(target / self.block_size)
        while len(table) < needed_blocks:
            table.append(self._alloc_block())
        self._logical_len[request_id] = target

    def share_prefix(self, src_id, dst_id, num_prefix_blocks):
        """Point dst at src's first N physical blocks (copy-on-write share of a common prefix)."""
        src_table = self.block_tables[src_id]
        shared = src_table[:num_prefix_blocks]
        for blk in shared:
            self.ref_count[blk] += 1
        self.block_tables[dst_id] = list(shared)
        self._logical_len[dst_id] = num_prefix_blocks * self.block_size

    def utilization(self, total_blocks):
        used = sum(len(t) for t in self.block_tables.values())
        # account for sharing: a physical block held by k requests is still ONE physical block
        physical_used = len({b for t in self.block_tables.values() for b in t})
        return physical_used, used, total_blocks


TOTAL_BLOCKS = 100
cache = PagedKVCache(TOTAL_BLOCKS, block_size=16)

# Request A processes a 40-token system prompt + 20 tokens of its own -> 60 tokens
cache.append_tokens("reqA", 60)
# Request B arrives with the SAME 40-token system prompt, then 8 of its own
cache.append_tokens("reqB", 8)
cache.share_prefix("reqA", "reqB", num_prefix_blocks=2)  # share the first 2 blocks (32 tokens) of prefix
cache.append_tokens("reqB", 40)                          # B keeps generating on top of the shared prefix

print("reqA block table:", cache.block_tables["reqA"])
print("reqB block table:", cache.block_tables["reqB"], "(first blocks shared with A)")
phys, logical, total = cache.utilization(TOTAL_BLOCKS)
print(f"\nLogical blocks across requests: {logical}")
print(f"Physical blocks actually used : {phys}  (sharing saved {logical - phys} block(s))")
print(f"Free physical blocks remaining: {total - phys} / {total}")


The block tables hold scattered physical block IDs, and `reqB` reuses `reqA`'s first blocks for free — no contiguous allocation, no wasted ceiling, and shared prefixes cost memory only once. That's the entire idea behind PagedAttention.

## Continuous batching: keep the GPU full

Paging solves *memory*. The second half of vLLM's speedup is about *scheduling*. The naive way to batch is **static batching**: gather B requests, run them together until the **longest** one finishes, then start the next batch. The problem is obvious once you picture it — a request that needs 20 tokens of output is stuck holding a GPU slot idle until its batch-mate that needs 800 tokens is done.

**Continuous batching** (a.k.a. iteration-level scheduling) instead makes admit/evict decisions *every decoding step*: the instant any request finishes, a waiting request takes its slot. The GPU stays full. Let's simulate both on the same workload.


In [ ]:
rng2 = np.random.default_rng(3)
N = 200
BATCH_SLOTS = 16                       # how many requests fit on the GPU at once
# Output lengths: again most short, a few long (the worst case for static batching).
output_lengths = np.clip(rng2.lognormal(mean=3.2, sigma=1.1, size=N), 4, 1024).astype(int)

def static_batching_steps(lengths, slots):
    """Each batch runs until its LONGEST member finishes; slots freed by short ones sit idle."""
    total_steps = 0
    for i in range(0, len(lengths), slots):
        batch = lengths[i:i+slots]
        total_steps += batch.max()     # the whole batch waits for the slowest
    return total_steps

def continuous_batching_steps(lengths, slots):
    """A finished request is instantly replaced; the GPU does sum(work)/slots steps."""
    return math.ceil(lengths.sum() / slots)

static_steps = static_batching_steps(output_lengths, BATCH_SLOTS)
cont_steps   = continuous_batching_steps(output_lengths, BATCH_SLOTS)

# "useful work" = total tokens to generate; slot-steps = capacity consumed
useful = output_lengths.sum()
static_gpu_util = useful / (static_steps * BATCH_SLOTS)
cont_gpu_util   = useful / (cont_steps * BATCH_SLOTS)

print(f"Workload: {N} requests, {BATCH_SLOTS} GPU slots, mean output {output_lengths.mean():.0f} tokens")
print(f"Static batching   : {static_steps:6,d} decode steps | GPU slot utilization {static_gpu_util:6.1%}")
print(f"Continuous batching: {cont_steps:6,d} decode steps | GPU slot utilization {cont_gpu_util:6.1%}")
print(f"\nThroughput speedup from continuous batching: {static_steps / cont_steps:.1f}x")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.bar(["Static", "Continuous"], [static_steps, cont_steps], color=["#C44E52", "#55A868"])
ax1.set_ylabel("Decode steps to clear the workload")
ax1.set_title("Lower is faster (same 200 requests)")
for i, v in enumerate([static_steps, cont_steps]):
    ax1.text(i, v, f"{v:,}", ha="center", va="bottom", fontweight="bold")

ax2.bar(["Static", "Continuous"], [static_gpu_util*100, cont_gpu_util*100], color=["#C44E52", "#55A868"])
ax2.set_ylabel("GPU slot utilization (%)")
ax2.set_title("How much of the GPU did useful work")
ax2.set_ylim(0, 100)
for i, v in enumerate([static_gpu_util*100, cont_gpu_util*100]):
    ax2.text(i, v + 2, f"{v:.0f}%", ha="center", fontweight="bold")
plt.tight_layout(); plt.show()


*Continuous batching clears the same 200 requests in far fewer steps because no slot waits on a batch-mate's long tail. The wider the spread in output lengths, the bigger the win — which is exactly the shape of real chat traffic.*

## Using vLLM for real (the OpenAI-compatible server)

In practice you almost never touch PagedAttention directly — you start vLLM's server and talk to it with the **OpenAI client you already know**. Two ways to use it:

**1. Offline batched inference (Python):**
```python
from vllm import LLM, SamplingParams
llm = LLM(model="meta-llama/Llama-3.1-8B-Instruct")      # loads weights, sets up paged KV cache
params = SamplingParams(temperature=0.7, max_tokens=128)
outputs = llm.generate(["Explain PagedAttention in one sentence."], params)
print(outputs[0].outputs[0].text)
```

**2. Serve an OpenAI-compatible HTTP endpoint (the production path):**
```bash
# one command starts a server with paging + continuous batching already on
vllm serve meta-llama/Llama-3.1-8B-Instruct --max-model-len 8192
```
Then point the standard OpenAI SDK at it — no code change beyond the `base_url`:


In [ ]:
# This cell makes a REAL call only if you have `vllm serve ...` running on localhost:8000.
# Otherwise it explains what you'd see, so the notebook still runs clean offline.
if HAS_VLLM_SERVER:
    from openai import OpenAI
    client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")  # vLLM ignores the key
    resp = client.chat.completions.create(
        model="meta-llama/Llama-3.1-8B-Instruct",
        messages=[{"role": "user", "content": "Explain PagedAttention in one sentence."}],
        max_tokens=80,
    )
    print(resp.choices[0].message.content)
else:
    print("[no local vLLM server] If one were running, the SAME OpenAI SDK from notebook 16")
    print("would work unchanged — only base_url points at localhost:8000 instead of api.anthropic")
    print("or api.openai. That drop-in compatibility is a big reason vLLM is the default")
    print("self-hosted serving engine: your client code, batching, and paging all stay invisible.")


## Where to actually run this (vLLM needs a Linux GPU)

vLLM requires CUDA + Linux, so it won't run on a Mac/Windows laptop — you need a Linux GPU somewhere. The two things to optimize for are **(a) does your editor attach to it** and **(b) who stops the meter when you walk away**. Cursor and VS Code are the same under the hood, so anything with **SSH access** works via *Remote-SSH* — that's most options below.

**VM-style GPUs (SSH in, attach Cursor/VS Code via Remote-SSH):**
- **RunPod** — per-minute billing, idle auto-terminate, a one-click vLLM template. The easiest EC2 replacement (~$0.30–0.50/hr for a 4090).
- **Vast.ai** — cheapest (a marketplace of other people's GPUs); great for learning, less reliable.
- **Lambda Labs** — clean, reliable A100/H100; more EC2-like (you remember to stop it).

**Notebook-but-actually-VS-Code-compatible:**
- **Lightning AI Studios** — persistent Linux env with real VS Code (browser *or* connect from local Cursor/VS Code), **sleeps when idle** so you don't pay for idle time, plus free monthly GPU hours. The best "dev box, not a notebook" option.
- **Kaggle Notebooks** — free T4/P100, ~30 hrs/week, no card needed (notebook-only, like Colab).

**Serverless / scale-to-zero (no idle cost by design — fixes the "pricey if you forget to stop it" problem):**
- **Modal** — write Python, it provisions a GPU per call and **scales to zero** when idle; has first-class examples that deploy an OpenAI-compatible vLLM endpoint in ~30 lines. Generous free credits.
- **RunPod Serverless / Replicate / Beam** — same pay-only-while-a-request-runs model.

**Don't even want to own a GPU?** The live cell above just needs an OpenAI-compatible `base_url`. Managed vLLM hosts — **Together, Fireworks, Anyscale, Baseten, OpenRouter** — run vLLM for you and hand you an endpoint; change one line and pay per token. (You won't *see* PagedAttention/batching internals, but the simulations in this notebook already showed those locally.)

**Quick pick:** for a persistent box with your editor on a budget, start with **Lightning AI** (free, auto-sleep) and graduate to **RunPod** for more power; to run vLLM as a real server with zero idle cost, use **Modal**.


## What else vLLM gives you (connections to earlier notebooks)

vLLM bundles the optimizations you've already met, all behind that one server command:
- **Prefix caching** — automatic reuse of shared prompt prefixes, the serving-side version of notebook 13's KV cache and notebook 16's "cold" cacheable layer.
- **Speculative decoding** (notebook 11) — pass `--speculative-config` to add a draft model; vLLM verifies its guesses in one pass, identical output distribution, lower latency.
- **Quantization** (notebook 10) — load AWQ/GPTQ/FP8 weights to fit bigger models or free more room for KV cache.
- **Tensor parallelism** — `--tensor-parallel-size N` shards one model across N GPUs when it doesn't fit on one.

The throughline: every trick in this curriculum for making a *single* generation cheaper becomes, at serving scale, a trick for fitting *more users* on the same GPU.

## Exercises


In [ ]:
# Exercise 1 (Warm-up): How much does block size matter?
# Task: Re-run the fragmentation simulation with BLOCK_SIZE set to 1, 16, 64, and 256.
#       Print paged utilization for each. Why does a SMALLER block raise utilization,
#       and what real cost are you trading away by shrinking it?
# Hint: Utilization rises as blocks shrink (less rounding-up waste), but tiny blocks mean
#       longer block tables and more per-step bookkeeping. Reuse `actual_lengths` as-is.

# YOUR CODE HERE


In [ ]:
# Exercise 2 (Apply): Model GQA's effect on how many users you can serve
# Task: Llama-2-70B actually uses Grouped-Query Attention (GQA) with 8 KV heads instead of 64.
#       That divides the KV cache by 8. Write kv_bytes_per_token_gqa(num_layers, d_model,
#       num_heads, num_kv_heads) and recompute "max tokens cached" for 70B on an 80GB A100,
#       with and without GQA. How many more concurrent 2048-token requests does GQA buy you?
# Hint: KV bytes/token = 2 * num_layers * (d_model * num_kv_heads / num_heads) * dtype_bytes.
#       For 70B use num_layers=80, d_model=8192, num_heads=64, num_kv_heads=8.

# YOUR CODE HERE


In [ ]:
# Exercise 3 (Extend): When does continuous batching stop helping?
# Task: Continuous batching wins most when output lengths VARY. Re-run the batching
#       simulation across sigma values [0.1, 0.5, 1.1, 2.0] for the lognormal output
#       lengths and plot the static/continuous speedup against sigma. What happens to the
#       speedup as all requests become the same length (sigma -> 0)?
# Hint: Wrap the two *_batching_steps calls in a loop over sigma, regenerate output_lengths
#       each time, collect static_steps/cont_steps, and plot. Expect the speedup -> ~1x
#       as variance vanishes (static batching is only wasteful when lengths differ).

# YOUR CODE HERE


<details>
<summary>Show solutions</summary>

```python
# Exercise 1
for bs in [1, 16, 64, 256]:
    blocks = np.ceil(actual_lengths / bs).astype(int)
    reserved = (blocks * bs).sum()
    print(f"block_size={bs:4d} -> utilization {used_tokens/reserved:6.1%}")
# Smaller blocks waste less on rounding (block_size=1 is ~100%), but each request's block
# table grows and the scheduler does more lookups per step. vLLM's default of 16 is the
# practical sweet spot between low fragmentation and low overhead.

# Exercise 2
def kv_bytes_per_token_gqa(num_layers, d_model, num_heads, num_kv_heads, dtype_bytes=2):
    return 2 * num_layers * (d_model * num_kv_heads // num_heads) * dtype_bytes

A100_80 = 80
weights_gb = 70 * 2
free_gb = A100_80 - weights_gb - 4   # bigger overhead margin for a 70B model
for tag, kv_heads in [("MHA (64 KV heads)", 64), ("GQA (8 KV heads)", 8)]:
    per_tok = kv_bytes_per_token_gqa(80, 8192, 64, kv_heads)
    max_tokens = max(free_gb, 0) * 1e9 / per_tok
    print(f"{tag:20s} {per_tok/1024:7.0f}KB/token  max cached {max_tokens:12,.0f} tokens "
          f"~{max_tokens/2048:6.0f} full 2048-tok requests")
# GQA cuts KV bytes/token 8x, so ~8x more concurrent requests fit — which is exactly why
# every modern large model (Llama-3, Mistral) ships with GQA.

# Exercise 3
sigmas = [0.1, 0.5, 1.1, 2.0]
speedups = []
for s in sigmas:
    lens = np.clip(np.random.default_rng(3).lognormal(3.2, s, N), 4, 1024).astype(int)
    speedups.append(static_batching_steps(lens, BATCH_SLOTS) / continuous_batching_steps(lens, BATCH_SLOTS))
plt.plot(sigmas, speedups, marker="o"); plt.xlabel("output-length spread (sigma)")
plt.ylabel("continuous / static speedup"); plt.title("Continuous batching wins more when lengths vary")
plt.show()
print(dict(zip(sigmas, [round(x, 2) for x in speedups])))
# As sigma -> 0 all requests are the same length, static batching wastes nothing, and the
# speedup collapses toward 1x. The benefit is entirely a function of length variance.
```
</details>


## Key Takeaways
- When you serve an LLM, the **KV cache — not the weights — is the scarce resource**: one OPT-13B token costs ~800 KB, so a 40 GB GPU holds only a few thousand tokens of cache across all users.
- Naive contiguous allocation reserves `max_len` per request and wastes 60–80% of memory to internal + external **fragmentation**; pre-vLLM systems ran at 20–40% utilization.
- **PagedAttention** applies OS virtual-memory paging to the KV cache — fixed-size blocks, per-request block tables, on-demand allocation — reaching ~96% utilization and enabling copy-on-write **sharing** of common prefixes.
- **Continuous batching** schedules at the per-step level so a finished request is instantly replaced, keeping the GPU full; the speedup scales with how much output lengths vary.
- In practice you get all of this from one `vllm serve` command and talk to it with the **unchanged OpenAI SDK** — and it composes with speculative decoding (11), quantization (10), and prefix caching (13).

## What's Next
Notebook **14 — RAG Fundamentals** moves from *how* you serve a model efficiently to *what* you put in its context window: retrieving external documents the model was never trained on, so its answers are grounded in your data rather than its memory.
